In [19]:
import pandas as pd
import pulp

from plots import plot_network
from utils import load_raw_data

# Set the Parameters

In [20]:
max_plants = 3
cost_per_mile = 1
cost_per_mile_over_threshold = 10
dist_threshold = 800  # cost = 1 for <= 800 and cost = 10 for > 800
multi_rate = True  # Above cost structure, if this parameter is True
open_map_in_cell = True  # This is for Jupyter Notebook. The map is saved in an HTML file anyway

# Load Input Data

In [21]:
def plot_input_map(input_df_dict):
    customers = input_df_dict['Customers']
    plants = input_df_dict['Plants']
    customers.rename(columns={'ID': 'Customer ID', 'Name': 'Customer Name'}, inplace=True)
    plants.rename(columns={'ID': 'Plant ID', 'Name': 'Plant Name'}, inplace=True)
    # Prepare data to plot the input network
    locations = prepare_location_dataframe(plants, customers)
    plot_network(locations, add_path=False, auto_open_map=False, title='Input Map',
                 open_map_in_cell=open_map_in_cell)

In [22]:
input_df_dict = load_raw_data(selected_sheets=['Customers', 'Demand', 'Plants', 'Distance Matrix'])

# Plot Input Map

In [23]:
def plot_input_map(input_df_dict):
    customers = input_df_dict['Customers']
    plants = input_df_dict['Plants']
    customers.rename(columns={'ID': 'Customer ID', 'Name': 'Customer Name'}, inplace=True)
    plants.rename(columns={'ID': 'Plant ID', 'Name': 'Plant Name'}, inplace=True)
    # Prepare data to plot the input network
    locations = prepare_location_dataframe(plants, customers)
    plot_network(locations, add_path=False, auto_open_map=False, title='Input Map',
                 open_map_in_cell=open_map_in_cell)

In [24]:
def prepare_location_dataframe(plants, customers):
    """Return a dataframe with the proper attributes needed for the plot."""
    origin_df = plants[['Plant Name', 'Latitude', 'Longitude']].copy()
    origin_df.rename(columns={'Plant Name': 'Origin', 'Latitude': 'Origin Lat',
                              'Longitude': 'Origin Lon'}, inplace=True)
    dest_df = customers[['Customer Name', 'Latitude', 'Longitude']].copy()
    dest_df.rename(columns={'Customer Name': 'Destination', 'Latitude': 'Dest Lat',
                            'Longitude': 'Dest Lon'}, inplace=True)
    df = pd.concat([origin_df, dest_df], sort=False, axis=1)
    return df

In [25]:
plot_input_map(input_df_dict)

# Network Optimization Block

In [26]:
def run_network_optimization(input_df_dict):
    """P-median problem"""
    # region Set Up Data
    customers = input_df_dict['Customers']
    plants = input_df_dict['Plants']
    customers.rename(columns={'ID': 'Customer ID', 'Name': 'Customer Name'}, inplace=True)
    plants.rename(columns={'ID': 'Plant ID', 'Name': 'Plant Name'}, inplace=True)
    demand = input_df_dict['Demand']
    dmd = demand.set_index(['Customer ID', 'Product ID', 'Time Period ID'])
    dist = input_df_dict['Distance Matrix'].set_index(['Plant ID', 'Customer ID'])
    if multi_rate:
        dist['rate'] = dist['Distance'].apply(
            lambda x: cost_per_mile * x if x < dist_threshold else cost_per_mile_over_threshold * x)
    else:
        dist['rate'] = dist['Distance'] * cost_per_mile

    # Sets
    cust = customers['Customer ID'].unique().tolist()
    plnt = plants['Plant ID'].unique().tolist()
    prod = demand['Product ID'].unique().tolist()
    peri = demand['Time Period ID'].unique().tolist()
    # endregion

    # region Optimization Model
    prob = pulp.LpProblem('run_network_optimization', pulp.LpMinimize)
    # Variables
    assign = pulp.LpVariable.dicts('y', indexs=[(i, j, k, t) for i in plnt for j in cust
                                                for k in prod for t in peri], cat=pulp.LpBinary)
    use_plant = pulp.LpVariable.dicts('x', indexs=[(i, t) for i in plnt for t in peri], cat=pulp.LpBinary)

    # Constraints
    # 1. Every customer - product - period must be assigned to one plant
    for j in cust:
        for k in prod:
            for t in peri:
                prob += pulp.lpSum(assign[i, j, k, t] for i in plnt) == 1

    # 2. No more than P plants
    prob += pulp.lpSum(use_plant) <= max_plants

    # 3. Set use_plant based on assign
    for (i, j, k, t), value in assign.items():
        prob += value <= use_plant[i, t]

    # KPI
    total_weighted_dist = pulp.lpSum(dist.loc[i, j]['Distance'] * dmd.loc[j, k, t]['Demand'] * value
                                     for (i, j, k, t), value in assign.items())
    # Objective function
    objective = pulp.lpSum((dist.loc[i, j]['rate'] * dmd.loc[j, k, t]['Demand'] * value)
                           for (i, j, k, t), value in assign.items())
    prob.setObjective(objective)
    status = prob.solve()
    # endregion

    # region Post Process
    print('=' * 40)
    if status == 1:
        print(f'Objective Value: {prob.objective.value():,.0f}')
        assigned_list = [k for k, v in assign.items() if v.varValue > 0.5]
        assigned_df = pd.DataFrame(assigned_list, columns=['Plant ID', 'Customer ID',
                                                           'Product ID', 'Time Period ID'])
        _opt_plants = [plants.loc[plants["Plant ID"] == k[0], "Plant Name"].iloc[0]
                       for k, v in use_plant.items() if v.varValue > 0.5]
        print(f'Selected plants are in: {", ".join(str(_) for _ in _opt_plants)}')
    else:
        raise ValueError("Optimization Failed")

    adf = pd.merge(assigned_df, dmd[['Demand']], how='inner',
                   left_on=['Customer ID', 'Product ID', 'Time Period ID'], right_index=True)
    adf = pd.merge(adf.set_index(['Plant ID', 'Customer ID']), dist[['Distance']],
                   how='inner', left_index=True, right_index=True).reset_index()
    num_customers = len(assigned_list)
    # # double checking the above value by calculating weighted avg dist
    # kpi = adf[['Plant ID', 'Distance']].groupby('Plant ID')['Distance'].agg(['mean', 'count', 'sum'])
    # # Just divide the total of kpi's "sum" columns by total of "count" columns
    adf['Within400'] = [True if x <= 400 else False for x in adf['Distance']]
    adf['Within800'] = [True if x <= 800 else False for x in adf['Distance']]
    adf['Within1200'] = [True if x <= 1200 else False for x in adf['Distance']]
    total_dmd = adf['Demand'].sum()
    weighted_avg_dist = total_weighted_dist.value() / total_dmd
    print(f'Weighted Average Distance: {weighted_avg_dist:,.1f}')
    # pct_cust_within400 = adf['Within400'].sum() / num_customers
    # pct_cust_within800 = adf['Within800'].sum() / num_customers
    # pct_cust_within1200 = adf['Within1200'].sum() / num_customers
    pct_dmd_within400 = adf.loc[adf['Within400'], 'Demand'].sum() / total_dmd
    pct_dmd_within800 = adf.loc[adf['Within800'], 'Demand'].sum() / total_dmd
    pct_dmd_within1200 = adf.loc[adf['Within1200'], 'Demand'].sum() / total_dmd
    print(f'Total demand: {total_dmd:,.0f}')
    print(f'Percentages of demand within 400 miles of a plant: {pct_dmd_within400 * 100:,.2f}%')
    print(f'Percentages of demand within 800 miles of a plant: {pct_dmd_within800 * 100:,.2f}%')
    print(f'Percentages of demand within 1200 miles of a plant: {pct_dmd_within1200 * 100:,.2f}%')
    print('=' * 40)

    # Create the set of lanes
    lanes = pd.merge(assigned_df, dmd[['Demand']], how='inner',
                     left_on=['Customer ID', 'Product ID', 'Time Period ID'], right_index=True)
    lanes['Lane'] = lanes['Plant ID'].map(str) + '-' + lanes['Customer ID'].map(str)
    lanes = lanes[['Plant ID', 'Customer ID', 'Demand', 'Lane']].merge(
        customers[['Customer ID', 'Customer Name', 'Latitude', 'Longitude']], on='Customer ID')
    lanes.rename(columns={'Latitude': 'Dest Lat', 'Longitude': 'Dest Lon'}, inplace=True)
    lanes = lanes.merge(plants[['Plant ID', 'Plant Name', 'Latitude', 'Longitude']], on='Plant ID')
    lanes.rename(columns={'Latitude': 'Origin Lat', 'Longitude': 'Origin Lon',
                          'Plant Name': 'Origin', 'Customer Name': 'Destination'}, inplace=True)
    lanes = lanes[['Lane', 'Origin', 'Destination', 'Demand', 'Plant ID',
                   'Origin Lat', 'Origin Lon', 'Customer ID', 'Dest Lat', 'Dest Lon']].drop_duplicates()
    # endregion
    # Plot output map
    plot_network(lanes, auto_open_map=False, title='Solution Map', open_map_in_cell=open_map_in_cell)

# Solve Optimization Model

In [27]:
run_network_optimization(input_df_dict)

Objective Value: 150,194,534,099
Selected plants are in: Detroit, New Orleans, Salt Lake City
Weighted Average Distance: 508.7
Total demand: 208,829,000
Percentages of demand within 400 miles of a plant: 26.43%
Percentages of demand within 800 miles of a plant: 97.19%
Percentages of demand within 1200 miles of a plant: 100.00%
